# Rice Leaf Disease Detection – PRCP-1001

# project team
# vaibhav shinde

## 1. Setup and Imports

In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16, MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import cv2, os, warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)
print("Libraries imported successfully")


c:\Users\user\miniconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Libraries imported successfully


## 2. Dataset Path & Classes

In [2]:

BASE_PATH = "C:\\Users\\user\\Downloads\\RiceLeaf_project\\Data"  # Change if needed
CLASSES = ['Bacterial leaf blight', 'Brown spot', 'Leaf smut']


## 3. Data Analysis

In [3]:
import os
import cv2


In [4]:

def collect_data_info(base_path, classes):
    data = []
    for c in classes:
        path = os.path.join(base_path, c)
        for img in os.listdir(path):
            if img.endswith(('jpg','png','jpeg')):
                p = os.path.join(path, img)
                im = cv2.imread(p)
                if im is not None:
                    data.append({
                        'class': c,
                        'height': im.shape[0],
                        'width': im.shape[1],
                        'size_kb': os.path.getsize(p)/1024
                    })
    return pd.DataFrame(data)

df = collect_data_info(BASE_PATH, CLASSES)
df.head()


,class,height,width,size_kb
0,Bacterial leaf blight,897,3081,408.133789
1,Bacterial leaf blight,897,3081,418.683594
2,Bacterial leaf blight,897,3081,486.936523
3,Bacterial leaf blight,897,3081,279.899414
4,Bacterial leaf blight,897,3081,222.955078


## 4. Data Generators & Augmentation

In [5]:

IMG_SIZE = (224,224)
BATCH_SIZE = 16

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_data = train_gen.flow_from_directory(
    BASE_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_data = train_gen.flow_from_directory(
    BASE_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)


Found 96 images belonging to 3 classes.
Found 23 images belonging to 3 classes.


## 5. Custom CNN Model

In [6]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True
    ),
    ModelCheckpoint(
        filepath='best_custom_cnn.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]


In [8]:
def custom_cnn():
    model = models.Sequential([
        layers.Conv2D(32,3,activation='relu',input_shape=(224,224,3)),
        layers.MaxPooling2D(),
        layers.Conv2D(64,3,activation='relu'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128,activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(3,activation='softmax')
    ])
    return model

cnn = custom_cnn()
cnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 186624)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    23,888,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,907,779 (91.20 MB)

 Trainable params: 23,907,779 (91.20 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# ================================
# TRAIN CUSTOM CNN
# ================================

cnn.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Training Custom CNN...")

history_cnn = cnn.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=callbacks
)

# Safe validation accuracy extraction
if 'val_accuracy' in history_cnn.history:
    cnn_val_accuracy = max(history_cnn.history['val_accuracy'])
    print(f"Custom CNN Best Validation Accuracy: {cnn_val_accuracy:.4f}")
else:
    print("Validation accuracy not available. Check validation_data.")


Training Custom CNN...
Epoch 1/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 559ms/step - accuracy: 0.3554 - loss: 10.6379
Epoch 1: val_accuracy improved from None to 0.34783, saving model to best_custom_cnn.h5



Epoch 1: finished saving model to best_custom_cnn.h5
6/6 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step - accuracy: 0.3438 - loss: 8.7379 - val_accuracy: 0.3478 - val_loss: 1.2914
Epoch 2/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 646ms/step - accuracy: 0.4059 - loss: 1.1803
Epoch 2: val_accuracy did not improve from 0.34783
6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 771ms/step - accuracy: 0.3438 - loss: 1.2001 - val_accuracy: 0.3478 - val_loss: 1.0974
Epoch 3/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 544ms/step - accuracy: 0.4398 - loss: 1.0959
Epoch 3: val_accuracy improved from 0.34783 to 0.39130, saving model to best_custom_cnn.h5



Epoch 3: finished saving model to best_custom_cnn.h5
6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 758ms/step - accuracy: 0.3854 - loss: 1.0969 - val_accuracy: 0.3913 - val_loss: 1.0965
Epoch 4/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 545ms/step - accuracy: 0.5062 - loss: 1.0799
Epoch 4: val_accuracy did not improve from 0.39130
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 655ms/step - accuracy: 0.4792 - loss: 1.0600 - val_accuracy: 0.2609 - val_loss: 1.1235
Epoch 5/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 557ms/step - accuracy: 0.3191 - loss: 1.1210
Epoch 5: val_accuracy did not improve from 0.39130
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 656ms/step - accuracy: 0.3646 - loss: 1.0762 - val_accuracy: 0.3913 - val_loss: 1.0923
Epoch 6/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 545ms/step - accuracy: 0.4092 - loss: 1.0483
Epoch 6: val_accuracy improved from 0.39130 to 0.56522, saving model to best_custom_cnn.h5



Epoch 6: finished saving model to best_custom_cnn.h5
6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 760ms/step - accuracy: 0.4583 - loss: 1.0349 - val_accuracy: 0.5652 - val_loss: 0.9493
Epoch 7/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 525ms/step - accuracy: 0.5373 - loss: 0.9734
Epoch 7: val_accuracy did not improve from 0.56522
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 636ms/step - accuracy: 0.5000 - loss: 0.9900 - val_accuracy: 0.4783 - val_loss: 0.9848
Epoch 8/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 567ms/step - accuracy: 0.4528 - loss: 0.9924
Epoch 8: val_accuracy did not improve from 0.56522
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 675ms/step - accuracy: 0.5417 - loss: 0.9426 - val_accuracy: 0.5217 - val_loss: 0.9682
Epoch 9/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 514ms/step - accuracy: 0.5569 - loss: 0.8873
Epoch 9: val_accuracy improved from 0.56522 to 0.65217, saving model to best_custom_cnn.h5



Epoch 9: finished saving model to best_custom_cnn.h5
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 725ms/step - accuracy: 0.5521 - loss: 0.9016 - val_accuracy: 0.6522 - val_loss: 0.8783
Epoch 10/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 490ms/step - accuracy: 0.5514 - loss: 0.8069
Epoch 10: val_accuracy did not improve from 0.65217
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 591ms/step - accuracy: 0.5208 - loss: 0.8327 - val_accuracy: 0.6087 - val_loss: 0.9213
Epoch 11/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 508ms/step - accuracy: 0.5564 - loss: 0.9177
Epoch 11: val_accuracy did not improve from 0.65217
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 603ms/step - accuracy: 0.6354 - loss: 0.8723 - val_accuracy: 0.4783 - val_loss: 1.1187
Epoch 12/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 496ms/step - accuracy: 0.6264 - loss: 0.7595
Epoch 12: val_accuracy did not improve from 0.65217
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 596ms/step - accuracy: 0.5833 - loss: 0.8230 - val_accuracy: 0.4783 - val_loss: 0.8757
Epoch 13/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 498ms/step - accuracy: 0.5880 - los

## 6. Transfer Learning – MobileNetV2

In [12]:

base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
base.trainable = False

model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(3,activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,339 (9.24 MB)

 Trainable params: 164,355 (642.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## 7. Training

In [13]:

# ================================
# TRAIN MOBILENETV2
# ================================

print("Training MobileNetV2...")

history_mobilenet = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=callbacks
)

# Store best validation accuracy
mobilenet_val_accuracy = max(history_mobilenet.history['val_accuracy'])

print(f"MobileNetV2 Best Validation Accuracy: {mobilenet_val_accuracy:.4f}")



Training MobileNetV2...
Epoch 1/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 425ms/step - accuracy: 0.3023 - loss: 1.5263
Epoch 1: val_accuracy improved from 0.65217 to 0.69565, saving model to best_custom_cnn.h5



Epoch 1: finished saving model to best_custom_cnn.h5
6/6 ━━━━━━━━━━━━━━━━━━━━ 10s 967ms/step - accuracy: 0.3021 - loss: 1.5107 - val_accuracy: 0.6957 - val_loss: 0.6459
Epoch 2/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 438ms/step - accuracy: 0.5521 - loss: 1.0007
Epoch 2: val_accuracy improved from 0.69565 to 0.95652, saving model to best_custom_cnn.h5



Epoch 2: finished saving model to best_custom_cnn.h5
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 634ms/step - accuracy: 0.6458 - loss: 0.8830 - val_accuracy: 0.9565 - val_loss: 0.3350
Epoch 3/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 456ms/step - accuracy: 0.6417 - loss: 0.8783
Epoch 3: val_accuracy did not improve from 0.95652
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 588ms/step - accuracy: 0.6667 - loss: 0.7504 - val_accuracy: 0.8696 - val_loss: 0.3108
Epoch 4/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 426ms/step - accuracy: 0.7568 - loss: 0.4825
Epoch 4: val_accuracy did not improve from 0.95652
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 554ms/step - accuracy: 0.8021 - loss: 0.4296 - val_accuracy: 0.8696 - val_loss: 0.4721
Epoch 5/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 465ms/step - accuracy: 0.8229 - loss: 0.3501
Epoch 5: val_accuracy did not improve from 0.95652
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 593ms/step - accuracy: 0.8229 - loss: 0.3596 - val_accuracy: 0.8696 - val_loss: 0.3536
Epoch 6/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 452ms/step - accuracy: 0.9281 - loss: 0.26

In [14]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": ["Custom CNN", "MobileNetV2"],
    "Best Validation Accuracy": [cnn_val_accuracy, mobilenet_val_accuracy]
})

comparison


,Model,Best Validation Accuracy
0,Custom CNN,0.652174
1,MobileNetV2,0.956522


## 8. Save Best Model

In [15]:

model.save("best_rice_disease_model.h5")
print("Model saved successfully")


Model saved successfully



## Challenges Faced and Solutions

### 1. Limited Dataset Size
- The dataset contains only 120 images (40 per class), which is very small for deep learning.
- **Solution:** Data augmentation techniques such as rotation, zooming, and horizontal flipping were applied to artificially increase data diversity and reduce overfitting.

### 2. Overfitting Risk
- Small datasets can cause models to memorize training data.
- **Solution:** Dropout layers and early stopping were used, along with transfer learning, to improve generalization.

### 3. Feature Extraction Difficulty
- Manually training a deep CNN from scratch may not capture complex disease patterns effectively.
- **Solution:** Transfer learning using MobileNetV2 pretrained on ImageNet was adopted.

### 4. Computational Constraints
- Training deep networks is resource-intensive.
- **Solution:** A lightweight architecture (MobileNetV2) was selected to balance accuracy and efficiency.

### 5. Model Selection for Production
- Multiple models may perform differently on the same dataset.
- **Solution:** Validation accuracy was compared between Custom CNN and MobileNetV2, and the best-performing model was selected for deployment.


## Project Completed Successfully